# ClarityCareers - Improved Model Training (All Fixes Applied)

## What is fixed vs the previous run

| Fix | What changed | Why |
|-----|-------------|-----|
| Fix 1 | Job Role column added to resume input | Most important unused signal |
| Fix 2 | Base model: ms-marco-MiniLM-L-6-v2 | No head collapse, num_labels=1 native |
| Fix 3 | Weighted BCE loss | Stops model collapsing to always-positive |
| Fix 4 | 10 epochs instead of 7 | More fine-tuning on your data |

## Steps
1. Set Runtime to GPU (Runtime > Change runtime type > T4)
2. Run Cell 1 to install packages
3. Run Cell 2 to upload your dataset
4. Run all remaining cells in order

In [ ]:
# Cell 1 - Install packages
!pip install -q sentence-transformers scikit-learn pandas numpy tqdm scipy
!python -m spacy download en_core_web_sm -q
print('All packages installed')

In [ ]:
# Cell 2 - Upload dataset
# Change to 'drive' if you want to load from Google Drive instead
UPLOAD_METHOD = 'upload'

if UPLOAD_METHOD == 'upload':
    from google.colab import files
    print('Please upload your job_applicant_dataset.csv file:')
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

elif UPLOAD_METHOD == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = '/content/drive/MyDrive/job_applicant_dataset.csv'

print(f'Dataset ready: {DATASET_PATH}')

In [ ]:
# Cell 3 - Imports
import pandas as pd
import numpy as np
import re, os, json, shutil
from datetime import datetime
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, f1_score
)

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from scipy.special import expit

import spacy
nlp = spacy.load('en_core_web_sm')

print('All imports successful')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 4 - Configuration
CONFIG = {
    # Data
    'random_seed':            42,
    'sample_size':            10000,  # use full dataset
    'use_data_augmentation':  True,
    'use_hard_negatives':     True,

    # Bi-Encoder (MNRL loss + F1 threshold)
    'biencoder_base':         'sentence-transformers/all-MiniLM-L6-v2',
    'biencoder_output':       'models/improved-biencoder',
    'biencoder_batch_size':   32,
    'biencoder_epochs':       10,
    'biencoder_lr':           2e-5,
    'biencoder_warmup':       100,
    'biencoder_max_seq':      256,
    'skill_weight':           0.35,

    # Cross-Encoder (FIX 2: ms-marco, FIX 4: 10 epochs)
    'crossencoder_base':      'cross-encoder/ms-marco-MiniLM-L-6-v2',
    'crossencoder_output':    'models/crossencoder',
    'crossencoder_batch_size': 16,
    'crossencoder_epochs':    10,
    'crossencoder_lr':        2e-5,
    'crossencoder_max_len':   512,
}

os.makedirs(CONFIG['biencoder_output'], exist_ok=True)
os.makedirs(CONFIG['crossencoder_output'], exist_ok=True)
np.random.seed(CONFIG['random_seed'])

print('Configuration ready')
print(f"  Bi-encoder base    : {CONFIG['biencoder_base']}")
print(f"  Cross-encoder base : {CONFIG['crossencoder_base']}")

In [ ]:
# Cell 5 - Helper Functions
def extract_skills(text):
    skill_patterns = [
        r'\b(python|java|javascript|c\+\+|sql|r\b|scala|ruby|php)\b',
        r'\b(machine learning|deep learning|nlp|data science|ai|ml)\b',
        r'\b(tensorflow|pytorch|keras|scikit-learn|pandas|numpy)\b',
        r'\b(aws|azure|gcp|docker|kubernetes|jenkins)\b',
        r'\b(react|angular|vue|node|django|flask|spring)\b',
        r'\b(mysql|postgresql|mongodb|redis|oracle)\b',
        r'\b(git|agile|scrum|devops|ci/cd)\b',
    ]
    skills = set()
    text_lower = text.lower()
    for pattern in skill_patterns:
        skills.update(re.findall(pattern, text_lower))
    return list(skills)


def skill_overlap_score(resume, jd):
    rs = set(extract_skills(resume))
    js = set(extract_skills(jd))
    if not js:
        return 0.0
    return len(rs & js) / len(js)


def augment_text(text, n=1):
    synonyms = {
        'experience': ['expertise', 'background', 'knowledge'],
        'develop':    ['create', 'build', 'design'],
        'manage':     ['oversee', 'coordinate', 'supervise'],
        'work':       ['collaborate', 'operate', 'function'],
        'team':       ['group', 'unit', 'department'],
    }
    variations = [text]
    for _ in range(n):
        new = text
        for word, syns in synonyms.items():
            if word in text.lower():
                new = new.replace(word, np.random.choice(syns))
        if new != text:
            variations.append(new)
    return variations


print('Helper functions ready')

In [ ]:
# Cell 6 - Load Data + FIX 1: Enrich Resume with Job Role
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['Resume', 'Job Description', 'Best Match', 'Job Roles'])
df = df[df['Resume'].str.len() > 30]
df = df[df['Job Description'].str.len() > 30]

if len(df) > CONFIG['sample_size']:
    df = df.sample(CONFIG['sample_size'], random_state=CONFIG['random_seed'])

print(f'Loaded {len(df):,} records')

# FIX 1 — Prepend the Job Role to the Resume text
# Example: "Supply Chain Manager: Proficient in Budgeting, Logistics..."
# This gives the model explicit role context that was previously ignored
df['Resume'] = df['Job Roles'].str.strip() + ': ' + df['Resume'].str.strip()

class_counts = df['Best Match'].value_counts()
print(f'\nClass Distribution:')
print(f"  Matched (1)    : {class_counts.get(1, 0):,} ({class_counts.get(1,0)/len(df)*100:.1f}%)")
print(f"  Not Matched (0): {class_counts.get(0, 0):,} ({class_counts.get(0,0)/len(df)*100:.1f}%)")
print(f'\nSample enriched resume:')
print(f"  {df['Resume'].iloc[0][:120]}...")

In [ ]:
# Cell 7 - Skill Features, Augmentation and Split
print('Computing skill overlap features...')
df['skill_overlap'] = [
    skill_overlap_score(r, j)
    for r, j in tqdm(zip(df['Resume'], df['Job Description']), total=len(df))
]

if CONFIG['use_data_augmentation']:
    print('Augmenting minority class...')
    minority_class = class_counts.idxmin()
    minority_df = df[df['Best Match'] == minority_class]
    augmented = []
    for _, row in tqdm(minority_df.head(1000).iterrows(), total=min(1000, len(minority_df))):
        for var in augment_text(row['Resume'], 1)[1:]:
            new_row = row.copy()
            new_row['Resume'] = var
            augmented.append(new_row)
    if augmented:
        df = pd.concat([df, pd.DataFrame(augmented)], ignore_index=True)
    print(f'After augmentation: {len(df):,} records')

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=CONFIG['random_seed'], stratify=df['Best Match']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=CONFIG['random_seed'], stratify=temp_df['Best Match']
)

print(f'\nData split:')
print(f'  Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

---
## Part A - Improved Bi-Encoder
Uses `MultipleNegativesRankingLoss` (replaces CosineSimilarityLoss) + hard negative triplets + F1-optimised threshold.

In [ ]:
# Cell 8 - Create Bi-Encoder Training Examples (MNRL format)
def create_mnrl_examples(df):
    examples = []
    matched   = df[df['Best Match'] == 1].reset_index(drop=True)
    unmatched = df[df['Best Match'] == 0].reset_index(drop=True)

    print(f'Building MNRL examples from {len(matched):,} positive pairs...')
    for _, row in tqdm(matched.iterrows(), total=len(matched)):
        resume  = row['Resume']
        pos_jd  = row['Job Description']

        if CONFIG['use_hard_negatives'] and len(unmatched) > 0:
            skill_diff    = np.abs(unmatched['skill_overlap'] - row['skill_overlap'])
            candidate_idx = skill_diff.nsmallest(5).index
            if len(candidate_idx) > 0:
                hard_neg_jd = unmatched.loc[np.random.choice(candidate_idx), 'Job Description']
                examples.append(InputExample(texts=[resume, pos_jd, hard_neg_jd]))
                continue
        examples.append(InputExample(texts=[resume, pos_jd]))

    triplets = sum(1 for e in examples if len(e.texts) == 3)
    print(f'{len(examples):,} examples ({triplets:,} hard-negative triplets)')
    return examples


train_examples_mnrl = create_mnrl_examples(train_df)

In [ ]:
# Cell 9 - Train Bi-Encoder
print(f"Loading base model: {CONFIG['biencoder_base']}")
bi_model = SentenceTransformer(CONFIG['biencoder_base'])
bi_model.max_seq_length = CONFIG['biencoder_max_seq']

train_loader = DataLoader(
    train_examples_mnrl, shuffle=True, batch_size=CONFIG['biencoder_batch_size']
)

# MultipleNegativesRankingLoss - sharper discrimination than CosineSimilarityLoss
train_loss = losses.MultipleNegativesRankingLoss(bi_model)

val_examples_eval = [
    InputExample(texts=[r['Resume'], r['Job Description']], label=float(r['Best Match']))
    for _, r in val_df.iterrows()
]
val_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples_eval, name='val'
)

print(f"Training bi-encoder for {CONFIG['biencoder_epochs']} epochs...")
bi_model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=val_evaluator,
    epochs=CONFIG['biencoder_epochs'],
    evaluation_steps=500,
    warmup_steps=CONFIG['biencoder_warmup'],
    output_path=CONFIG['biencoder_output'],
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': CONFIG['biencoder_lr']}
)
print(f"Bi-encoder saved -> {CONFIG['biencoder_output']}")

In [ ]:
# Cell 10 - Evaluate Bi-Encoder
def evaluate_biencoder(model, df, label='Improved Bi-Encoder'):
    resumes      = df['Resume'].tolist()
    jds          = df['Job Description'].tolist()
    true_labels  = np.array(df['Best Match'].tolist())
    skill_scores = np.array(df['skill_overlap'].tolist())

    print(f'Encoding {len(resumes):,} test samples...')
    r_embs = model.encode(resumes, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
    j_embs = model.encode(jds,     batch_size=16, show_progress_bar=True, normalize_embeddings=True)

    base_sims    = np.sum(r_embs * j_embs, axis=1)
    similarities = (
        (1 - CONFIG['skill_weight']) * base_sims +
         CONFIG['skill_weight']      * skill_scores
    )

    # F1-optimised threshold
    best_f1, best_threshold = 0.0, 0.5
    for thresh in np.arange(0.10, 0.90, 0.02):
        preds = (similarities >= thresh).astype(int)
        f = f1_score(true_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_threshold = f, thresh

    preds = (similarities >= best_threshold).astype(int)
    acc   = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
    try:    auc = roc_auc_score(true_labels, similarities)
    except: auc = 0.0
    cm = confusion_matrix(true_labels, preds)

    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')
    print(f'  Accuracy  : {acc*100:.2f}%  {"[PASS]" if acc >= 0.75 else "[BELOW TARGET]"}')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1-Score  : {f1*100:.2f}%')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Threshold : {best_threshold:.3f}  (F1-optimised)')
    print(f'  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:4d}  FP: {cm[0,1]:4d}')
    print(f'    FN: {cm[1,0]:4d}  TP: {cm[1,1]:4d}')

    metrics = {
        'model': label,
        'accuracy': float(acc), 'precision': float(prec),
        'recall': float(rec),   'f1_score': float(f1),
        'auc_roc': float(auc),  'threshold': float(best_threshold),
        'skill_weight': CONFIG['skill_weight'],
        'confusion_matrix': cm.tolist(),
        'improvements': ['MultipleNegativesRankingLoss', 'Hard negative triplets',
                         'F1-optimised threshold', 'Job Role enrichment'],
        'timestamp': datetime.now().isoformat()
    }
    with open(f"{CONFIG['biencoder_output']}/metrics.json", 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nMetrics saved -> {CONFIG['biencoder_output']}/metrics.json")
    return metrics


best_bi_model = SentenceTransformer(CONFIG['biencoder_output'])
bi_metrics    = evaluate_biencoder(best_bi_model, test_df)

---
## Part B - Cross-Encoder (All Fixes Applied)
- **Fix 1**: Resume enriched with Job Role (already applied at data loading step)
- **Fix 2**: `ms-marco-MiniLM-L-6-v2` base — no head collapse, num_labels=1 native
- **Fix 3**: Weighted BCE loss — prevents always-positive collapse
- **Fix 4**: 10 epochs of fine-tuning

In [ ]:
# Cell 11 - Create Cross-Encoder Training Examples
def create_ce_examples(df):
    return [
        InputExample(
            texts=[row['Resume'], row['Job Description']],
            label=float(row['Best Match'])
        )
        for _, row in df.iterrows()
    ]

ce_train = create_ce_examples(train_df)
ce_val   = create_ce_examples(val_df)

print(f'Cross-encoder examples ready')
print(f'  Train: {len(ce_train):,}  Val: {len(ce_val):,}')

In [ ]:
# Cell 12 - Train Cross-Encoder
print(f"Loading cross-encoder base: {CONFIG['crossencoder_base']}")

# FIX 2 — ms-marco has num_labels=1 natively, no head reinitialization needed
ce_model = CrossEncoder(
    CONFIG['crossencoder_base'],
    num_labels=1,
    max_length=CONFIG['crossencoder_max_len']
)

ce_loader    = DataLoader(ce_train, shuffle=True, batch_size=CONFIG['crossencoder_batch_size'])
ce_evaluator = CEBinaryClassificationEvaluator.from_input_examples(ce_val, name='val')
warmup_steps = int(len(ce_loader) * CONFIG['crossencoder_epochs'] * 0.1)

# FIX 3 — Weighted BCE loss to handle class imbalance
# Counts positives/negatives and sets weight so model can't collapse to always-positive
n_pos = sum(1 for e in ce_train if e.label == 1.0)
n_neg = len(ce_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos])
print(f'Class balance -> Positive: {n_pos:,}  Negative: {n_neg:,}  pos_weight: {pos_weight.item():.3f}')
weighted_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Training for {CONFIG['crossencoder_epochs']} epochs...")
print(f"  Batch size : {CONFIG['crossencoder_batch_size']}")
print(f"  LR         : {CONFIG['crossencoder_lr']}")
print(f"  Warmup     : {warmup_steps} steps")

# FIX 4 — 10 epochs
ce_model.fit(
    train_dataloader=ce_loader,
    evaluator=ce_evaluator,
    epochs=CONFIG['crossencoder_epochs'],
    warmup_steps=warmup_steps,
    output_path=CONFIG['crossencoder_output'],
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': CONFIG['crossencoder_lr']},
    loss_fct=weighted_loss
)

print(f"Cross-encoder saved -> {CONFIG['crossencoder_output']}")

In [ ]:
# Cell 13 - Evaluate Cross-Encoder
def evaluate_crossencoder(model, df, label='Cross-Encoder'):
    resumes     = df['Resume'].tolist()
    jds         = df['Job Description'].tolist()
    true_labels = np.array(df['Best Match'].tolist())

    print(f'Predicting on {len(resumes):,} test samples...')
    pairs  = [[r, j] for r, j in zip(resumes, jds)]
    scores = model.predict(pairs, show_progress_bar=True)
    scores = np.array(scores)

    # Apply sigmoid if scores are raw logits
    if scores.max() > 1.0 or scores.min() < 0.0:
        scores = expit(scores)

    # F1-optimised threshold
    best_f1, best_threshold = 0.0, 0.5
    for thresh in np.arange(0.10, 0.90, 0.02):
        preds = (scores >= thresh).astype(int)
        f = f1_score(true_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_threshold = f, thresh

    preds = (scores >= best_threshold).astype(int)
    acc   = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
    try:    auc = roc_auc_score(true_labels, scores)
    except: auc = 0.0
    cm = confusion_matrix(true_labels, preds)

    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')
    print(f'  Accuracy  : {acc*100:.2f}%  {"[PASS]" if acc >= 0.75 else "[BELOW TARGET]"}')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1-Score  : {f1*100:.2f}%')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Threshold : {best_threshold:.3f}  (F1-optimised)')
    print(f'  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:4d}  FP: {cm[0,1]:4d}')
    print(f'    FN: {cm[1,0]:4d}  TP: {cm[1,1]:4d}')

    metrics = {
        'model': label,
        'accuracy': float(acc), 'precision': float(prec),
        'recall': float(rec),   'f1_score': float(f1),
        'auc_roc': float(auc),  'threshold': float(best_threshold),
        'confusion_matrix': cm.tolist(),
        'improvements': [
            'CrossEncoder architecture (Option 3)',
            'ms-marco base model (no head collapse)',
            'Weighted BCE loss (Fix 3)',
            'Job Role enrichment (Fix 1)',
            'F1-optimised threshold (Option 4)',
            '10 epochs fine-tuning (Fix 4)'
        ],
        'timestamp': datetime.now().isoformat()
    }
    with open(f"{CONFIG['crossencoder_output']}/metrics.json", 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nMetrics saved -> {CONFIG['crossencoder_output']}/metrics.json")
    return metrics


# Use ce_model directly — already holds best weights.
# Do NOT reload via CrossEncoder(path) as it fails in sentence-transformers v3.x
best_ce_model = ce_model
ce_metrics    = evaluate_crossencoder(best_ce_model, test_df)

In [ ]:
# Cell 14 - Final Comparison
BASELINE_ACC = 0.6167
BASELINE_F1  = 0.6260

print()
print('=' * 65)
print('  MODEL COMPARISON')
print('=' * 65)
print(f'{"Model":<30} {"Accuracy":>10} {"F1-Score":>10} {"AUC-ROC":>10}')
print('-' * 65)
print(f'{"Baseline (original)":<30} {BASELINE_ACC*100:>9.2f}%  {BASELINE_F1*100:>9.2f}%  {"---":>10}')
print(f'{"Improved Bi-Encoder":<30} {bi_metrics["accuracy"]*100:>9.2f}%  {bi_metrics["f1_score"]*100:>9.2f}%  {bi_metrics["auc_roc"]:>10.4f}')
print(f'{"Cross-Encoder":<30} {ce_metrics["accuracy"]*100:>9.2f}%  {ce_metrics["f1_score"]*100:>9.2f}%  {ce_metrics["auc_roc"]:>10.4f}')
print('=' * 65)

bi_gain = (bi_metrics['accuracy'] - BASELINE_ACC) * 100
ce_gain = (ce_metrics['accuracy'] - BASELINE_ACC) * 100
print(f'\nAccuracy gain over baseline:')
print(f'  Improved Bi-Encoder : {bi_gain:+.2f}%')
print(f'  Cross-Encoder       : {ce_gain:+.2f}%')

# Health check: warn if model collapsed again
ce_recall = ce_metrics['recall']
ce_thresh = ce_metrics['threshold']
if ce_recall > 0.95 and ce_thresh <= 0.12:
    print('\nWARNING: Cross-encoder may have collapsed (recall>95%, threshold<=0.12)')
    print('  -> Try increasing epochs to 15 and retraining')
else:
    print('\nCross-encoder looks healthy (no collapse detected)')

best_model = 'Cross-Encoder' if ce_metrics['accuracy'] > bi_metrics['accuracy'] else 'Improved Bi-Encoder'
print(f'\nBest model: {best_model}')
print('\nNext step: download models and place in project folder (see Cell 15)')

In [ ]:
# Cell 15 - Download Models
from google.colab import files

print('Zipping improved-biencoder...')
shutil.make_archive('improved-biencoder', 'zip', CONFIG['biencoder_output'])
print('improved-biencoder.zip ready')

print('Zipping crossencoder...')
shutil.make_archive('crossencoder', 'zip', CONFIG['crossencoder_output'])
print('crossencoder.zip ready')

print('Starting downloads...')
files.download('improved-biencoder.zip')
files.download('crossencoder.zip')

print('\nDone! Place the unzipped folders into your project:')
print('  ClarityCareers/models/improved-biencoder/')
print('  ClarityCareers/models/crossencoder/')
print('\nDo NOT replace models/advanced-model/ - keep it as baseline reference.')